# Epistemic Driven Epistemic Landscapes (EDEL) Pipeline

This notebook demonstrates the full modular EDEL pipeline, from data collection to interactive 3D landscapes.

## 0. Parameters and setup

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

# EDEL Pipeline Imports
from edel.config.defaults import RUN_CONFIG
from edel.io.llm import get_llm_client
from edel.pipeline import (
    run_data_stage, run_structuring_stage, run_embedding_stage,
    run_projection_stage, run_vector_field_stage, run_clustering_stage,
    run_labeling_stage, run_landscape_stage
)
from edel.viz import (
    plot_abstract_length_dist, plot_publication_year_dist, plot_citation_dist,
    plot_projection_2d, plot_transition_signatures, plot_movement_magnitudes,
    plot_vector_field, plot_field_magnitude, plot_field_density,
    plot_clusters_on_landscape, plot_field_clusters, plot_cluster_trajectories,
    print_cluster_summaries, plot_epistemic_map,
    plot_landscape_3d, plot_landscape_contour
)

# Environment Setup
# os.environ["OPENAI_API_KEY"] = "your-key-here"

config = RUN_CONFIG.copy()
print("✅ Setup complete.")

## 1. Data Collection

In [ ]:
print("Running Stage 1: Data Collection...")
df = run_data_stage(config)
print(f"Loaded {len(df)} documents.")

plot_abstract_length_dist(df)
plot_publication_year_dist(df)
plot_citation_dist(df)

## 2. Structuring

In [ ]:
print("Running Stage 2: Structuring abstracts...")
df = run_structuring_stage(df, config)
df.head(2)

## 3. Text Embeddings

In [ ]:
print("Running Stage 3: Embedding aspects...")
df = run_embedding_stage(df, config)
print("Embeddings computed.")

## 4. Dimensionality Reduction

In [ ]:
print("Running Stage 4: Projection...")
df = run_projection_stage(df, config)

method = config["dimensionality_reduction"]["method"]
plot_projection_2d(df, method=method, draw_arrows=True, arrow_step=20)
plot_transition_signatures(df)
plot_movement_magnitudes(df)

## 5. Vector Field

In [ ]:
print("Running Stage 5: Vector Field...")
field = run_vector_field_stage(df, config)

plot_vector_field(field, field_type="total", color_by_mag=True)
plot_field_magnitude(field, operator="pm")
plot_field_density(field)

## 6. Clustering

In [ ]:
print("Running Stage 6: Clustering...")
df, field = run_clustering_stage(df, field, config)

plot_clusters_on_landscape(df, cluster_key="domain")
plot_field_clusters(field, cluster_key="field")
plot_cluster_trajectories(df, cluster_key="style")

## 7. Labeling

In [ ]:
print("Running Stage 7: Labeling...")
llm_client = get_llm_client(config.get("labeling", {}))
label_results = run_labeling_stage(df, field, config, llm_client)

print_cluster_summaries(label_results, cluster_key="domain")

## 8. Landscape

In [ ]:
print("Running Stage 8: Landscape preparation...")
landscape_results = run_landscape_stage(df, field, config)

topic_name = config["data"]["provider"]["topic_name"]

# Final Epistemic Map
plot_epistemic_map(df, label_results, topic_name=topic_name)

# Interactive 2D Contour Map
plot_landscape_contour(df, landscape_results, field=field, topic_name=topic_name)

# Interactive 3D Surface
plot_landscape_3d(df, landscape_results, topic_name=topic_name)